### Project 3 Part 1

#### Data and model preparation

In [ ]:
from huggingface_hub import login
HF_TOKEN = "ENTER YOUR TOKEN HERE" 
login(HF_TOKEN)

In [2]:
!pip install -q transformers accelerate pydantic torch outlines
!pip install -q trl peft accelerate bitsandbytes
!pip install -q transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00:00:0100:01


In [3]:
from datasets import load_dataset
import random
import os
import re
import json
import math
import torch
import gc
from datetime import datetime
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig
from google.colab import userdata

# Model helper
def load_model(model_name, lora_path=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )

    if lora_path:
        model = PeftModel.from_pretrained(model, lora_path)

    model.eval()
    return model, tokenizer

def cleanup(*objects):
    """Delete model/tokenizer objects and free GPU memory."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [4]:
# Consts
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = (
    "You are a careful math solver. Think about the following problem step by step.\n"
    "At the end, output exactly ONE line with the final answer in LaTeX, strictly follows the format:\n"
    "\\boxed{<integer>}\n"
    "Please only include integer in the boxed"
    "Do not include any other text after the boxed answer. \n"
    "Do not generate any boxed content other than the final answer. \n"
)

In [5]:
#load model and dataset
from datasets import load_dataset
test_ds = load_dataset("gsm8k", "main")["test"]
model, tokenizer = load_model(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

#### Task 1.1

In [6]:
# Question 1
# ── Answer extraction ──
def extract_boxed(text):
    """Extract content from the last \\boxed{...}, handling nested braces."""
    idx = text.rfind(r"\boxed{")
    if idx == -1:
        return None
    else:
        i = idx + len(r"\boxed{") #start from the end of last \boxed
        stack = ["{"]
        start = i
        while i < len(text):
            if text[i] == "{":
                stack.append("{")
            elif text[i] == "}":
                stack.pop()
                if not stack:
                    return text[start:i] #if all braces have corresponding match, return
            i += 1
        
    return None


def extract_ground_truth(raw_answer, gt_format):
    """Extract the ground-truth answer string from the dataset's answer field."""
    if gt_format == "hashmarks":
        m = re.search(r"####\s*(.+)", raw_answer)
        if m:
            return m.group(1).strip().replace(",", "")
        return raw_answer.strip()
    if gt_format == "boxed":
        boxed = extract_boxed(raw_answer)
        if boxed is not None:
            return boxed.strip()
        return raw_answer.strip()
    return raw_answer.strip()


def extract_model_answer(text):
    extracted = extract_boxed(text)
    if extracted:
        return extracted.strip().replace(",", "")
    # Final/Answer marker
    m = re.search(r"(Answer|Final)\s*:\s*([-]?\d[\d,]*)", text, flags=re.IGNORECASE)
    if m:
        return m.group(2).strip().replace(",", "")

    # fallback: last integer
    nums = re.findall(r"-?\d+", text.replace(",", ""))
    if nums:
        return nums[-1]
    return None

# ── Prompt building & batched generation ──

def build_prompts(tokenizer, questions):
    """Build chat-formatted prompt strings for a list of questions."""
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )
    return prompts


def generate_batch(model, tokenizer, questions):
    """Generate responses for a batch of questions in one forward pass."""
    prompts = build_prompts(tokenizer, questions)
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True
    ).to(model.device)

    # modified
    prompt_lens = inputs["attention_mask"].sum(dim=1).tolist()

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)

    responses = []
    for i in range(len(questions)):
        new_tokens = out[i][prompt_lens[i]:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return responses


# ── Evaluation runner ──

def evaluate_gsm8k(model, tokenizer, test_ds, num_samples=100, batch_size=16, seed=42):
    """Zero-shot eval on GSM8K test set. Returns accuracy and records."""

    # fixed subset
    idxs = list(range(len(test_ds)))
    random.Random(seed).shuffle(idxs)
    idxs = idxs[:num_samples] # randomly shuffle the indices to obtain num_samples indices
    subset = test_ds.select(idxs) #select 100 samples by default

    questions = [ex["question"] for ex in subset] #questions
    gts = [extract_ground_truth(ex["answer"], gt_format="hashmarks") for ex in subset] #groundtruths

    correct = 0
    records = []

    for start in tqdm(range(0, num_samples, batch_size)):
        batch_q = questions[start:start+batch_size] #each batch, take idxs from start to start + batch size
        batch_gt = gts[start:start+batch_size]

        batch_resp = generate_batch(model, tokenizer, batch_q) #build prompt, tokenizer, model.generate, decode

        for q, gt, resp in zip(batch_q, batch_gt, batch_resp):
            pred = extract_model_answer(resp)
            ok = (pred is not None and str(pred).strip() == str(gt).strip())# same numerical answer
            correct += int(ok)
            records.append({
                "question": q,
                "gt": gt,
                "pred": pred,
                "correct": ok,
                "raw_output": resp
            })

    acc = correct / num_samples
    return acc, records



In [7]:
print("Test size:", len(test_ds))

base_acc, base_records = evaluate_gsm8k(
    model,
    tokenizer,
    test_ds,
    num_samples=100,
    batch_size=16,
    seed=42
)


print(f"\nBase model zero-shot accuracy: {base_acc:.2%}")

Test size: 1319


100%|██████████| 7/7 [06:35<00:00, 56.56s/it]


Base model zero-shot accuracy: 36.00%


In [8]:
print(base_records)

[{'question': 'Jared is trying to increase his typing speed. He starts with 47 words per minute (WPM). After some lessons the next time he tests his typing speed it has increased to 52 WPM. If he continues to increase his typing speed once more by 5 words, what will be the average of the three measurements?', 'gt': '52', 'pred': '3', 'correct': False, 'raw_output': " average of the three measurements?\nassistant\nTo find the average of the three measurements, we first need to determine the total number of words Jared types over these three sessions.\n\nThe first session was at 47 WPM for an unknown duration, let's call this duration \\( t_1 \\).\nThe second session was at 52 WPM for an unknown duration, let's call this duration \\( t_2 \\).\nThe third session was at 57 WPM for an unknown duration, let's call this duration \\( t_3 \\).\n\nSince the total number of words typed is equal to the product of the word rate and the duration for each session, we can write:\n\n\\[ 47t_1 = 52t_2 =

**Answer Q1**
- Base model zero-shot accuracy: 36.00%

In [10]:
# Question 2
wrong = [r for r in base_records if not r["correct"]]
for i in range(3):
    print("-----------")
    print(f"Question: {wrong[i]["question"]}")
    print(f"Ground Truth: {wrong[i]["gt"]}")
    print(f"Prediction: {wrong[i]["pred"]}")
    print(f"Raw output: {wrong[i]["raw_output"]}")

-----------
Question: Jared is trying to increase his typing speed. He starts with 47 words per minute (WPM). After some lessons the next time he tests his typing speed it has increased to 52 WPM. If he continues to increase his typing speed once more by 5 words, what will be the average of the three measurements?
Ground Truth: 52
Prediction: 3
Raw output:  average of the three measurements?
assistant
To find the average of the three measurements, we first need to determine the total number of words Jared types over these three sessions.

The first session was at 47 WPM for an unknown duration, let's call this duration \( t_1 \).
The second session was at 52 WPM for an unknown duration, let's call this duration \( t_2 \).
The third session was at 57 WPM for an unknown duration, let's call this duration \( t_3 \).

Since the total number of words typed is equal to the product of the word rate and the duration for each session, we can write:

\[ 47t_1 = 52t_2 = 57t_3 \]

We want to find 

**Answer Q2**:

- **Example 1**:
    - **Question:** Jared’s typing speed increased from 47 WPM to 52 WPM, and then to 57 WPM. What is the average of the three measurements?

    - **Model excerpt:** We want to find the average of the three measurements, which means we need to calculate the sum of all the durations divided by 3: 
    $$
    \text{Average} = \frac{t_1 + t_2 + t_3}{3}
    $$ However, since we don't have enough information to solve for \( t_1 \), \( t_2 \), or \( t_3 \) individually, the model concludes that it cannot compute the exact value.

    - **Failure explanation:** 
        - **Type:** reasoning/logical error and misunderstanding the problem
        - The problem only asks us to average the three measured speeds (47 + 52 + 57) / 3. However, the model overcomplicated the problem by assuming exact time durations are necessary to solve this problem.

    - **Extracted Answer:** 3 (fallback: extracted the last integer)
    - **Ground Truth:** 52

- **Example 2**:
    - **Question:** A wooden bridge can carry no more than 5000 pounds. A delivery truck filled with identical boxes, each weighing 15 pounds, will pass over the bridge. The combined weight of the driver and the empty truck is 3755 pounds. What is the maximum number of boxes which can be loaded onto the truck while not exceeding the bridge's weight limit?

    - **Model excerpt:**  Divide the remaining capacity by the weight of one box to find the maximum number of boxes:
    $$
    \text{Maximum number of boxes} = \frac{1245}{15} = 249
    $$

    - **Failure explanation:** 
        - **Type**: Arithmetic slip
        - The reasoning process is correct, but the model made an arithemetic mistake: 1245 / 15 = 83, not 249

    - **Extracted Answer:** 249
    - **Ground Truth:** 83

- **Example 3**:
    - **Question:** Dominick went to his team's changing room and saw half as many robots as helmets and half as many helmets as footballs kept there. If there were 20 helmets, calculate the total number of items Dominick saw.
     - **Model excerpt:** 
    $$
    \text{Number of robots} = \frac{\text{Number of helmets}}{2}
    = \frac{20}{2}
    = 10
    $$

    Next, since the number of helmets is half the number of footballs:

    $$
    \text{Number of footballs}
    = 2 \times \text{Number of helmets}
    = 2 \times 20
    = 40
    $$

    Now, we add up all the items:

    $$
    \text{Total number of items}
    = \text{Number of robots}
    + \text{Number of helmets}
    + \text{Number of footballs}
    $$

    $$
    \text{Total number of items}
    = 10 + 20 + 40
    = 72
    $$

    - **Failure explanation:** 
        - **Type**: Arithmetic slip
        - The reasoning process is correct, but the model made an arithemetic mistake: 10 + 20 + 40 = 70 not 72
    
    - **Extracted Answer:** 72
    - **Ground Truth:** 70

- **Recurring patterns:** 
    - The model sometimes introduce redundant variables and assumptions
    - Arithmetic errors are very common in wrong answers, even when the reasoning steps are all correct. Multi-step calculations are especially prone to errors. 

